# 01 Observing: Reading Model Internals During Generation

**Scenario**: You have trained a linear probe (or a sparse autoencoder, SAE) that extracts a signal from hidden states—sentiment, refusal tendency, or hallucination cues. Offline validation looks great, and now you want to know: **can we run it token-by-token inside a real serving engine and watch that signal live?** This is the deployment question behind most interpretability and safety monitoring work (probe monitoring, SAE feature tracking).

vLLM only returns output tokens; intermediate hidden states stay on the GPU, interleaved across concurrent requests and captured inside CUDA graphs. Attaching a Python hook either fails to fire, forces you to disable CUDA graphs, or leaves you manually matching rows back to requests.

In this tutorial, you will:

1. Extract per-token final hidden states from the engine and train a sentiment probe;
2. Write a workload that runs the probe on every generated token and exports scores to the host;
3. Run two requests concurrently, watch scores evolve token by token, and verify that **observation neither alters outputs nor disables CUDA graphs**;
4. See what changes when swapping the linear probe for an SAE encoder.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

import tempfile
from collections import defaultdict

import torch
from safetensors.torch import save_file
from torch import nn
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)
PROBE = os.path.join(tempfile.gettempdir(), "sentiment_probe.safetensors")  # Trained probe weights

## Step 1: Extract Training Data and Train the Probe

The probe reads the **final hidden state**: the vector after the last layer and final norm, right before the LM head—equivalent to `hidden_states[-1]` in Hugging Face. If you already have an offline-trained probe, it reads that exact same representation and can be dropped straight in.

Here we train one on the fly. The dataset has 20 short sentences with sentiment labels, where every token counts as a training sample, so we need to extract the final hidden state for **every token** of those sentences directly from vLLM using a lightweight observation workload:

- `pllm.final_hidden(tokens="all")` provides one row for every token computed in the current step, including prompt tokens;
- A single step computes at most `max_num_batched_tokens` (512 here) tokens, so we set `max_rows=512` on the export;
- `overflow="block"` makes the export wait rather than drop rows if the delivery queue fills up, since training needs every row;
- Alongside `identity`, each row carries its `position` within the sequence, which we use to reassemble tokens back into their original sentences.

We will dive into export mechanics in Step 2. Once collected, an `nn.Linear(896, 1)` trained with logistic regression finishes in a few seconds; we save its weights as a safetensors file for the Step 2 workload to load.

In [2]:
positive = ["I love this movie, it is wonderful.", "What a fantastic and joyful day.",
            "The food was delicious and the staff were kind.", "I am so happy with my new phone.",
            "This book is a delight to read.", "Our trip was amazing and fun.", "She gave a brilliant, inspiring talk.",
            "The concert was beautiful.", "Thank you, this made my week!", "We won the game and everyone cheered."]
negative = ["I hate this movie, it is awful.", "What a terrible and miserable day.",
            "The food was disgusting and the staff were rude.", "I am so angry with my new phone.",
            "This book is a chore to read.", "Our trip was a disaster.", "She gave a boring, painful talk.",
            "The concert was horrible.", "This ruined my whole week.", "We lost the game and everyone cried."]
hidden: dict[tuple[int, int], torch.Tensor] = {}  # (sentence index, token position) -> final hidden state


class EveryToken(pllm.Workload):
    @pllm.export(name="final_hidden", max_rows=512, max_cols=896, overflow="block")
    def rows(self, h: torch.Tensor = pllm.final_hidden(tokens="all")) -> torch.Tensor:
        return h.float()

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, position, row in zip(event.identities, event.positions, event.tensor()):
            hidden[identity.prompt_index, position] = row


with pllm.make_llm(CONFIG, workload=EveryToken()) as llm:
    outputs = llm.generate(positive + negative, SamplingParams(max_tokens=1), use_tqdm=False)
# Keep only prompt tokens; the engine may have also computed one generated token, which we ignore.
rows = [torch.stack([hidden[index, position] for position in range(len(output.prompt_token_ids))])
        for index, output in enumerate(outputs)]
features = torch.cat(rows).cuda()
labels = torch.cat([torch.full((len(r),), float(i < len(positive))) for i, r in enumerate(rows)]).cuda()

torch.manual_seed(0)
probe = nn.Linear(features.shape[1], 1).cuda()
optimizer = torch.optim.Adam(probe.parameters(), lr=1e-2)
for _ in range(300):
    loss = nn.functional.binary_cross_entropy_with_logits(probe(features).squeeze(-1), labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
accuracy = ((probe(features).squeeze(-1) > 0) == (labels > 0.5)).float().mean().item()
print(f"{len(features)} 个 token 样本，训练集准确率 {accuracy:.2f}")

save_file({"w": probe.weight.detach()[0].cpu(), "b": probe.bias.detach().cpu()}, PROBE)

155 个 token 样本，训练集准确率 0.87


## Step 2: Write the Observation Workload

The workload declares three pieces:

- `probe`: A state holding the probe's `w` (length = hidden width) and `b`. With no optimizer attached, it is read-only; `init=pllm.Pretrained(...)` points to the file we just saved so the engine loads `w` and `b` by field name at startup. States are stored in the model dtype (bf16), so we cast to float32 before computing.
- `score`: Your math. Parameter **default values** declare where inputs come from: `pllm.final_hidden()` gives the current token's final hidden state (one row per active request), and `pllm.parameter("probe", "w")` reads a field from state.
- `@pllm.export(...)`: Declares where the result goes. Return tensors are copied to CPU and delivered in order to `on_export`. Exports are bounded: at most `max_rows` rows and `max_cols` columns per step, and by default events are dropped rather than blocking if the queue fills up, ensuring observation never stalls the GPU or exhausts host memory.

Each event received by `on_export` carries the step ID, `identities` mapping each row in the step to its request, and the tensor itself. We collect scores per request.

In [3]:
scores: dict[int, list[float]] = defaultdict(list)


class SentimentMonitor(pllm.Workload):
    probe = pllm.InstanceState(
        {"w": pllm.Field(lambda width, _: (width,)), "b": (1,)}, init=pllm.Pretrained(PROBE)
    )

    @pllm.export(name="sentiment", max_rows=4, max_cols=1)
    def score(self, h: torch.Tensor = pllm.final_hidden(), w: torch.Tensor = pllm.parameter("probe", "w"),
              b: torch.Tensor = pllm.parameter("probe", "b")) -> torch.Tensor:
        return torch.sigmoid(h.float() @ w.float()[:, None] + b.float())

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, value in zip(event.identities, event.tensor()[:, 0].tolist()):
            scores[identity.prompt_index].append(value)

## Step 3: Run Two Requests Concurrently

One request writes about a wonderful day at the beach; the other writes about a terrible day when everything went wrong. They generate side-by-side in the same batch, and their probe scores should sit near 1 and 0 respectively. Let's run plain vLLM first as a baseline, then generate the same prompts with our monitor attached.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["Write two sentences about a wonderful day at the beach.",
             "Write two sentences about a terrible day when everything went wrong."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=48, temperature=0.0)

with pllm.make_llm(CONFIG) as plain:
    baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]

with pllm.make_llm(CONFIG, workload=SentimentMonitor()) as llm:
    texts = [output.outputs[0].text for output in llm.generate(prompts, greedy)]
    eager_reason = llm.diagnostics().eager_reason

for index, text in enumerate(texts):
    values = scores[index]
    strip = "".join("+" if v > 0.5 else "-" for v in values)
    print(f"请求 {index}：{len(values)} 个 token，平均分 {sum(values) / len(values):.2f}")
    print(f"    {strip}")
    print(f"    {text!r}")
print("输出与原生 vLLM 相同:", texts == baseline)
print("eager 原因:", repr(eager_reason), "（空字符串表示模型仍在 CUDA graph 里运行）")

请求 0：47 个 token，平均分 0.98
    +++++++++++++++++++++++-+++++++++++++++++++++++
    'On a wonderful day at the beach, the warm sun rays caress the sand, and the sound of waves crashing against the shore fills the air. The salty air invigorates the senses, and the feeling of being surrounded by nature and'
请求 1：38 个 token，平均分 0.05
    ----------------+--------------------+
    'A terrible day for me was when I had to cancel a major project due to unforeseen circumstances, leaving me with no choice but to focus on my personal life and prioritize my health.'
输出与原生 vLLM 相同: True
eager 原因: '' （空字符串表示模型仍在 CUDA graph 里运行）


Each `+` or `-` shows the probe's prediction on a single generated token. Although both requests share the same batch, their scores stay cleanly separated per request: the beach story is almost all `+`, and the bad-day story is almost all `-`. The last two lines confirm the two guarantees of observation: outputs match plain vLLM bit-for-bit, and `eager_reason` is empty, meaning the model stays inside CUDA graphs with the probe captured alongside it.

## Step 4: Swapping in an SAE Encoder

SAE monitoring follows the exact same pattern, just wider: store the encoder matrix `W_enc` (dictionary size × hidden width) and bias in state, and replace the math with `ReLU(h @ W_enc.T + b)`. Because dictionaries often have tens of thousands of features—exceeding export width limits—export the indices of the top-k most active features per token rather than the full vector:

```python
@pllm.export(name="sae_top8", max_rows=4, max_cols=8)
def encode(self, h: torch.Tensor = pllm.final_hidden(), w_enc: torch.Tensor = pllm.parameter("sae", "w_enc"),
           b_enc: torch.Tensor = pllm.parameter("sae", "b_enc")) -> torch.Tensor:
    return torch.relu(h.float() @ w_enc.T + b_enc).topk(8, dim=-1).indices.float()
```

You can also change which layer is read: `pllm.hidden(L)` gives the newly computed delta at layer `L`—the MLP output before adding back to the residual stream, matching the "MLP out" site used in work like Llama Scope—whereas `pllm.final_hidden()` is the vector after the final norm.

## Summary

- Observation workload = a read-only state + a method decorated with `@pllm.export`; parameter defaults declare what to read.
- Exports are bounded, tagged with request identity, and delivered in order to `on_export`.
- Observation never alters model outputs and does not disable CUDA graphs.

## Common Mistakes

The two most common mistakes in observation workloads come from treating workload methods like ordinary Python functions: printing values inside a method, and averaging across the row dimension.

In [5]:
class Chatty(pllm.Workload):
    @pllm.export(name="norm", max_rows=4, max_cols=1)
    def norm(self, h: torch.Tensor = pllm.final_hidden()) -> torch.Tensor:
        print("范数", h.float().norm().item())  # Trying to inspect a value inside the method
        return h.float().norm(dim=-1, keepdim=True)


class Centered(pllm.Workload):
    @pllm.export(name="norm", max_rows=4, max_cols=1)
    def norm(self, h: torch.Tensor = pllm.final_hidden()) -> torch.Tensor:
        centered = h.float() - h.float().mean(0)  # Trying to subtract a "mean activation", but averaging over dim 0 (requests)
        return centered.norm(dim=-1, keepdim=True)


try:
    pllm.explain(Chatty(), CONFIG)
except RuntimeError as error:
    print("Chatty:", error)
try:
    testing.check(Centered(), CONFIG)
except testing.ContractError as error:
    print("Centered:", error.failures[0])

Chatty: Tensor.item() cannot be called on meta tensors
Centered: [row permutation] request 'r2', export norm at token 5: max |diff| 23.652. method `norm` mixes rows: equal inputs for this request produced different outputs. Dimension 0 holds request rows; reduce features (dim=-1), not requests (dim=0).


**Why `.item()` and `print` fail.** When building the engine, pLLM traces your method using shape-only meta tensors (which hold no numerical values—hence `"meta tensors"`) and captures it into CUDA graphs alongside the model. During generation, the graph is replayed directly and your Python lines do not execute step-by-step; even if they did, copying scalars from GPU to CPU every step would stall the entire engine. To inspect values, return them from an `@pllm.export` method—that is what export is for. For the same reason, Python branches on tensor values like `if h.norm() > 1:` are invalid; use `torch.where` instead.

**Why you must not average over dimension 0.** Each row along dimension 0 is a different request, ordered by the scheduler. `h.mean(0)` leaks other requests into your result and causes a request's score to change depending on who it happens to share a step with. `testing.check` catches this directly by permuting rows and swapping co-scheduled requests. If you need a fixed "mean activation," compute it offline and store it in state just like probe weights.

## Self-Check

1. Your probe was trained offline on Hugging Face's `hidden_states[13]` (the residual stream after layer 12). Which input marker should you read in pLLM?

<details><summary>Answer</summary>

`pllm.residual(12)`. `pllm.hidden(12)` is only layer 12's MLP output before the residual addition, and `pllm.final_hidden()` is the vector after the final norm—all three are distinct quantities.
</details>

2. Why do outputs stay identical to plain vLLM even with the monitoring workload attached?

<details><summary>Answer</summary>

Exports are read-only: they neither modify logits nor mutate state, and the model continues running inside CUDA graphs (`eager_reason` is empty), so the model executes the exact same computation as without a workload.
</details>

3. You pass 8 prompts to `generate` at once while using an export with `max_rows=4` reading `tokens="prefill"`. What happens?

<details><summary>Answer</summary>

Prompts finishing prefill in the same step share those 4 rows; excess rows are dropped (counted in the event's `dropped_rows`) without blocking generation. To guarantee one row per prompt, send at most `max_rows` prompts per `generate` call or increase `max_rows`. In this tutorial, `CONFIG` allows at most 4 concurrent requests, so a step has at most 4 prefill-end rows.
</details>

Next up, [02 Per-Request LoRA](02_request_adapters.ipynb) starts modifying the model: once each request finishes reading its prompt, a hypernetwork generates a dedicated LoRA adapter for it on the fly.